In [18]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, round, lit, count, stddev, when

# 1. Alustetaan Spark MongoDB-tuella
spark = SparkSession.builder \
    .appName("PISA-Finland-Analysis") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.4.0") \
    .getOrCreate()

uri_2022 = "mongodb://127.0.0.1:27017/pisa_database.year2022"
df_2022 = spark.read.format("mongodb").option("spark.mongodb.read.connection.uri", uri_2022).load()

In [15]:
def get_finland_stats(year):
    uri = f"mongodb://127.0.0.1:27017/pisa_database.year{year}"
    df = spark.read.format("mongodb").option("spark.mongodb.read.connection.uri", uri).load()
    
    fin_df = df.filter(col("CNT") == "FIN")
    
    # Choosing the correct ICT column based on the year
    ict_col_name = "ICTAVHOM_SUM" if "ICTAVHOM_SUM" in df.columns else "ICTAVHOM"
    
    # Averages and rounding directly in Spark
    stats = fin_df.select(
        round(avg("PV1MATH"), 2).alias("avg_math"),
        round(avg(ict_col_name), 2).alias("avg_ict")
    ).collect()[0]
    
    return stats

try:
    print("\n--- PISA ANALYSIS: FINLAND 2012 vs 2022 ---\n")
    
    stats_2012 = get_finland_stats(2012)
    stats_2022 = get_finland_stats(2022)
    
    print(f"Year 2012 (Mathematics): {stats_2012['avg_math']} points")
    print(f"Year 2012 (ICT Availability): {stats_2012['avg_ict']} (WLE index)")
    print("-" * 40)
    print(f"Year 2022 (Mathematics): {stats_2022['avg_math']} points")
    print(f"Year 2022 (ICT Availability): {stats_2022['avg_ict']} (sum variable)")
    
    # Calculating change
    change = stats_2022['avg_math'] - stats_2012['avg_math']
    print(f"\nChange in mathematics proficiency: {change:.2f} points.")

except Exception as e:
    print(f"Error in analysis: {e}")



--- PISA ANALYSIS: FINLAND 2012 vs 2022 ---

Year 2012 (Mathematics): 507.53 points
Year 2012 (ICT Availability): 38.49 (WLE index)
----------------------------------------
Year 2022 (Mathematics): 475.28 points
Year 2022 (ICT Availability): 5.67 (sum variable)

Change in mathematics proficiency: -32.25 points.


In [16]:
def get_correlation(year):
    uri = f"mongodb://127.0.0.1:27017/pisa_database.year{year}"
    df = spark.read.format("mongodb").option("spark.mongodb.read.connection.uri", uri).load()
    
    # 1. Valitaan oikea sarake dynaamisesti
    # Tarkistetaan löytyykö _SUM versio, muuten käytetään perusversiota
    if "ICTAVHOM_SUM" in df.columns:
        ict_col = "ICTAVHOM_SUM"
    elif "ICTAVHOM" in df.columns:
        ict_col = "ICTAVHOM"
    else:
        print(f"Varoitus: Vuoden {year} datasta ei löytynyt ICTAVHOM-saraketta.")
        return None

    # 2. Suodatetaan Suomi ja varmistetaan, ettei valitussa sarakkeessa ole tyhjiä
    # Huom: Käytetään col(ict_col) muuttujaa merkkijonon sijasta
    fin_df = df.filter(
        (col("CNT") == "FIN") & 
        (col(ict_col).isNotNull()) & 
        (col("PV1MATH").isNotNull())
    )
    
    # 3. Lasketaan korrelaatio dynaamisesti valitulla sarakkeella
    try:
        correlation = fin_df.stat.corr(ict_col, "PV1MATH")
        return correlation
    except Exception as e:
        print(f"Virhe korrelaation laskennassa vuodelle {year}: {e}")
        return None

corr_2012 = get_correlation(2012)
corr_2022 = get_correlation(2022)

print(f"\n--- CORRELATION ANALYSIS (ICT vs. MATHEMATICS) ---")
print(f"Correlation in 2012: {corr_2012:.3f}")
print(f"Correlation in 2022: {corr_2022:.3f}")


--- CORRELATION ANALYSIS (ICT vs. MATHEMATICS) ---
Correlation in 2012: 0.067
Correlation in 2022: 0.117


In [19]:
def analyze_math_by_time(year):
    uri = f"mongodb://127.0.0.1:27017/pisa_database.year{year}"
    df = spark.read.format("mongodb").option("spark.mongodb.read.connection.uri", uri).load()
    
    if year == 2012:
        # 2012 käyttää raakoja kategorioita (1-7)
        df = df.withColumn("Usage_Group", 
            when(col("ICTWKDY").isin(1, 2, 3), "1. Low (0-1h)")
            .when(col("ICTWKDY").isin(4, 5), "2. Moderate (1-4h)")
            .when(col("ICTWKDY").isin(6, 7), "3. Heavy (4h+)")
            .otherwise(None))
    else:
        # 2022 käyttää WLE-indeksiä (liukulukuja)
        # Säädetään rajat vastaamaan suurin piirtein samaa jakaumaa
        df = df.withColumn("Usage_Group", 
            when(col("ICTWKDY") < -0.5, "1. Low (Below Avg)")
            .when((col("ICTWKDY") >= -0.5) & (col("ICTWKDY") <= 1.0), "2. Moderate (Avg)")
            .when(col("ICTWKDY") > 1.0, "3. Heavy (High Use)")
            .otherwise(None))

    stats = df.filter((col("CNT") == "FIN") & (col("Usage_Group").isNotNull())) \
        .groupBy("Usage_Group") \
        .agg(
            round(avg("PV1MATH"), 1).alias("Avg_Math"),
            # Lisätään määrä, jotta nähdään ryhmien koko
            count("PV1MATH").alias("N_Students") 
        ).orderBy("Usage_Group")
    
    return stats

# 3. Suoritetaan analyysi
try:
    print("\n--- MATEMAATTINEN TAITO VS. INTERNET-AIKA SUOMESSA ---")
    print("Vuosi 2012:")
    analyze_math_by_time(2012).show()
    
    print("\nVuosi 2022:")
    analyze_math_by_time(2022).show()

except Exception as e:
    print(f"Virhe analyysissä: {e}")


--- MATEMAATTINEN TAITO VS. INTERNET-AIKA SUOMESSA ---
Vuosi 2012:
+------------------+--------+----------+
|       Usage_Group|Avg_Math|N_Students|
+------------------+--------+----------+
|     1. Low (0-1h)|   521.6|      1910|
|2. Moderate (1-4h)|   512.0|      5309|
|    3. Heavy (4h+)|   496.5|      1257|
+------------------+--------+----------+


Vuosi 2022:
+-------------------+--------+----------+
|        Usage_Group|Avg_Math|N_Students|
+-------------------+--------+----------+
| 1. Low (Below Avg)|   499.6|      1728|
|  2. Moderate (Avg)|   493.3|      5451|
|3. Heavy (High Use)|   434.2|      1056|
+-------------------+--------+----------+



In [20]:
def check_math_correlation(year):
    uri = f"mongodb://127.0.0.1:27017/pisa_database.year{year}"
    df = spark.read.format("mongodb").option("spark.mongodb.read.connection.uri", uri).load()
    
    # Suodatetaan Suomi ja poistetaan tyhjät (nullit sotkevat korrelaation)
    fin_df = df.filter(
        (col("CNT") == "FIN") & 
        (col("ICTWKDY").isNotNull()) & 
        (col("PV1MATH").isNotNull())
    )
    
    # Lasketaan korrelaatio
    r = fin_df.stat.corr("ICTWKDY", "PV1MATH")
    
    # Lasketaan myös hajonta, joka kertoo ryhmän sisäisistä eroista
    std_dev = fin_df.select(stddev("PV1MATH")).collect()[0][0]
    
    return r, std_dev

try:
    r12, sd12 = check_math_correlation(2012)
    r22, sd22 = check_math_correlation(2022)
    
    print("\n--- MATEMAATTINEN TAITO VS. INTERNET-AIKA SUOMESSA ---")
    print(f"Vuosi 2012: r = {r12:.3f} (Hajonta: {sd12:.1f})")
    print(f"Vuosi 2022: r = {r22:.3f} (Hajonta: {sd22:.1f})")
    
    # Lasketaan korrelaation muutos prosentteina "selitysvoimasta" (r^2)
    print(f"\nAnalyysi: Korrelaatio on muuttunut {((r22-r12)/abs(r12))*100:.1f} %")

except Exception as e:
    print(f"Virhe: {e}")


--- MATEMAATTINEN TAITO VS. INTERNET-AIKA SUOMESSA ---
Vuosi 2012: r = -0.240 (Hajonta: 89.9)
Vuosi 2022: r = -0.198 (Hajonta: 89.8)

Analyysi: Korrelaatio on muuttunut 17.7 %
